In [ ]:
import rebound as rb
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
import pymcel as pym
import plotly.graph_objects as go
import plotly.express as px

Aqui definire el parametro de tiempo por ahora

In [ ]:
date ='2029-04-13 18:00:00'

UA2km = 149597870.7
yr2s = 365.25 * 24 * 3600
Msun2kg = 1.98847e30

# **Simulación**

In [ ]:
# Definimos la simulacion del sistema solar con Rebound
sim_solar = rb.Simulation()

nombres_JPL = ['301', 'Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune']

sim_solar.units = ('yr', 'AU', 'Msun')

sim_solar.add('399',date=date)
sim_solar.add('Apophis',date=date)
sim_solar.add('Sun',date=date)

In [ ]:
from importlib import reload
import funciones_utiles

reload(funciones_utiles)

from funciones_utiles import integracion_Apo_tierr, distancia_minima

T = 0.0009 # Años de simulación'
paso = 20 # Paso de integración en minutos 
pasoy = paso/(24*365.25*60) # Paso de integración en años (un tercio de día)
ts = np.arange(0, T, pasoy)




_, _, rs, vs, E = integracion_Apo_tierr(sim_solar, ts)

In [ ]:
0.0009*365.25*24

In [ ]:
rs_Raro, vs_Raro, dis, ts_min, ts_d = distancia_minima(sim_solar, ts, date, nombres_JPL)

### **Gráfica**

In [ ]:
nombres = ['Tierra', 'Apophis', 'Sol', 'Luna']
color = ['blue', 'red', 'yellow', 'gray']

plt.figure(figsize=(10, 8))
for i in range( 3- 1):  # Excluimos el Sol para la gráfica
    plt.plot(rs_Raro[i, :, 0], rs_Raro[i, :, 1], label=f'{nombres[i]}', color=f'{color[i]}')

#plt.plot(rs_Raro[3, :, 0] - rs_Raro[0, :, 0], rs_Raro[3, :, 1] - rs_Raro[0, :, 1], color='red', label='Luna (mínima distancia)')
plt.scatter(rs[0, 0, 0], rs[0, 0, 1], color='blue', marker='o', s=100, label='Tierra (inicio)')
plt.xlabel('X (AU)')
plt.ylabel('Y (AU)')
plt.title('Trayectorias de los cuerpos en el sistema solar respecto a la Tierra')
plt.legend()
plt.axis('equal')
plt.grid()
plt.show()


In [ ]:
dis_min_luna = np.min(np.sqrt((rs_Raro[3, :, 0] - rs_Raro[0, :, 0] - (rs_Raro[1, :, 0]-rs_Raro[0, :, 0]))**2 + (rs_Raro[3, :, 1] - rs_Raro[0, :, 1]- (rs_Raro[1, :, 1]-rs_Raro[0, :, 1]))**2))

dis_min_luna_km = dis_min_luna * UA2km
print(f'Distancia mínima a la Luna: {dis_min_luna:.6f} AU ({dis_min_luna_km:.2f} km)')

In [ ]:
dis[-1] - 6378

### **Animaciones**

In [ ]:
# Animacion 3D con Matplotlib: trayectoria de Apophis respecto a la Tierra (marco geocentrico)
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

r_apo_tierra = rs[1] - rs[0]  # vector posicion Apophis - Tierra
step = 1  # menos frames para reducir tamano del HTML
idx = np.arange(0, len(ts), step)

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')

line, = ax.plot([], [], [], color='crimson', lw=2, label='Apophis (rel. Tierra)')
point = ax.scatter([], [], [], color='crimson', s=30)
ax.scatter(0, 0, 0, color='royalblue', s=60, label='Tierra (origen)')

max_range = np.max(np.abs(r_apo_tierra)) * 1.1
ax.set_xlim(-max_range, max_range)
ax.set_ylim(-max_range, max_range)
ax.set_zlim(-max_range, max_range)
ax.set_xlabel('x [AU]')
ax.set_ylabel('y [AU]')
ax.set_zlabel('z [AU]')
ax.set_title('Animacion 3D de Apophis respecto a la Tierra (sin Sol)')
ax.legend()

def update(frame):
    k = idx[frame]
    x = r_apo_tierra[:k+1, 0]
    y = r_apo_tierra[:k+1, 1]
    z = r_apo_tierra[:k+1, 2]

    line.set_data(x, y)
    line.set_3d_properties(z)
    point._offsets3d = ([r_apo_tierra[k, 0]], [r_apo_tierra[k, 1]], [r_apo_tierra[k, 2]])
    return line, point

ani = FuncAnimation(fig, update, frames=len(idx), interval=60, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


In [ ]:
# animcion 2D 
from funciones_utiles import animacion_2d_tierra_apophis

fig2, anim2 = animacion_2d_tierra_apophis(
    rs_Raro[:2],
    intervalo_ms=100, 
    fecha_inicial=date,
    ts=ts,
    utc_offset_horas=-5,
    show = False
)


In [ ]:
anim2.save('animacion_apophis_tierra_ventana_pequeña.gif', writer='pillow', fps=150)

# **Consulta de Efemerides de JPL**

In [ ]:

from datetime import datetime, timedelta


end_ts = ts[-1]
t_iso = datetime.fromisoformat(date)
t_end = t_iso + end_ts*timedelta(days= 365.25)

print(f"Fecha inicial: {date}")
print(f"Fecha final: {t_end.strftime('%Y-%m-%d %H:%M:%S')}")


fecha = dict(start=date, stop=t_end.strftime('%Y-%m-%d %H:%M:%S'), step=f'{paso}m')
print(fecha)

_, _, X_apo = pym.consulta_horizons(id = 'Apophis', location = '@0', epochs = fecha)

len(X_apo), len(ts), len(rs_Raro[1])

In [ ]:
ts_min

Gráfica de la trayectoria real de Apophis con respecto al baricentro del sistema solar

In [ ]:
rs_JPL_Apo = np.column_stack((X_apo['x'], X_apo['y'], X_apo['z']))/(UA2km*1000)
rs_JPL_Apo.shape

In [ ]:
plt.figure(figsize=(8, 8))

plt.plot(rs_JPL_Apo[:, 0], rs_JPL_Apo[:, 1], color='green', label='Apophis (JPL)')
plt.plot(rs_Raro[1, :, 0], rs_Raro[1, :, 1], color='blue', label='Apophis (Rebound)')
plt.scatter(0, 0, color='yellow', s=100, label='Sol')
plt.xlabel('X (AU)')
plt.ylabel('Y (AU)')
plt.title('Comparación de la órbita de Apophis: JPL vs Rebound')
plt.legend()
plt.grid()
plt.show()

NameError: name 'plt' is not defined

### **Error de la integracion con respecto a JPL**

In [ ]:
diff_integracion_apo = np.linalg.norm(rs_Raro[1] - rs_JPL_Apo, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(ts*12, diff_integracion_apo*UA2km, color='magenta')
plt.xlabel('Tiempo (años)')
plt.ylabel('Diferencia de posición (km)')
plt.title('Diferencia entre la posición de Apophis en Rebound y JPL')
plt.grid()
plt.show()


# **Exploremos la dinamica de este sistema**

En esta parte del notebook, dada la integración con N cuerpos para la estimacion de la distancia minima de Apophis con respecto a la tierra, se procedera a dar un analisis de su orbita de forma dinamica aplicando la teoria de N cuerpos, 2 cuerpos y 3 cuerpos.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(nombres_JPL, dis, color='purple', s=100, label='Distancia Mínima Apophis-Tierra')
plt.xlabel('Planetas')
plt.ylabel('Distancia Mínima (km)')
plt.title('VAriacion de la distancia minima entre Apophis y la tierra, agregando mas cuerpos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.legend()
plt.grid()
plt.show()


In [ ]:
UA2km = 149597870.7
yr2s = 365.25 * 24 * 3600

# comparacion de velocidades a lo largo del tiempo
plt.figure(figsize=(10, 6))
plt.plot(ts, np.linalg.norm(vs_Raro[0]*UA2km/yr2s, axis=1), label='Velocidad Tierra', color='blue')
plt.plot(ts, np.linalg.norm(vs_Raro[1]*UA2km/yr2s, axis=1), label='Velocidad Apophis', color='red')
plt.xlabel('Tiempo (años)')
plt.ylabel('Velocidad (km/s)')
# plt.xlim(0.05, 0.061)  # Enfocamos en la ventana alrededor del encuentro cercano
plt.title('Comparación de velocidades entre Tierra y Apophis')
plt.legend()
plt.grid()     
plt.show()  

In [ ]:

print(f"La velocidad promedio de la Tierra es: {np.mean(np.linalg.norm(vs_Raro[0]*UA2km/yr2s, axis=1)):.4f} km/s")
print(f"La velocidad promedio de Apophis es: {np.mean(np.linalg.norm(vs_Raro[1]*UA2km/yr2s, axis=1)):.4f} km/s")

## **Analisis de Cuadraturas N cuerpos**

In [ ]:
# Añadir los demas cuerpos

for i in range(len(nombres_JPL)):
    sim_solar.add(nombres_JPL[i], date=date)
    

Momentum lineal $\vec{p}$ de el sistema:

$$
\vec{P}_{tot} = \sum_i m_i \dot{\vec{r}}_i
$$

In [ ]:
Msun2kg = 1.98847e30

In [ ]:
ms = np.array([sim_solar.particles[i].m for i in range(len(sim_solar.particles))])

P_i = ms[:, np.newaxis, np.newaxis] * vs_Raro

P_tots_vec = np.sum(P_i, axis=0)      # Convertir a kg*km/s

P_tots = np.linalg.norm(P_tots_vec, axis=1)

print('El momento lineal total del sistema es:', P_tots_vec[0]* Msun2kg*UA2km/yr2s, 'kg*km/s')
print('y su magnitud es:', P_tots[0]* Msun2kg*UA2km/yr2s, 'kg*km/s')


Sigamos con la segunda cuadratura y es la posicion del vector inicial asociado al centro de masa $R_0$ definido de la forma:

$$
\vec{R_0} = \vec{R}_{CM} - \vec{V}_{CM} t 
$$



In [ ]:
M = np.sum(ms)

mi_ri = ms[:, np.newaxis, np.newaxis] * rs_Raro
R_cm_vec =  np.sum(mi_ri, axis = 0) / M  # Posición del centro de masa
V_cm_vec = P_tots_vec / M  # Velocidad del centro de masa



R_0 = R_cm_vec - V_cm_vec * (ts- ts[0])[:, np.newaxis]

print('El vector posición del centro de masa en el tiempo inicial es:', (R_0[0])* (UA2km), 'km') 
print('y su magnitud es:', np.linalg.norm(R_0[0])* (UA2km), 'km')  


Cuadratura de momentum angular $\vec{L}'_{tot}$ (Vector del plano invariante de Laplace del sistema):

$$
\sum_i m_i \vec{r}_i \times \dot{\vec{r}}_i + M\vec{R}_{CM} \times \vec{V}_{CM} = \vec{L}_{tot}
$$

In [ ]:
L_i = np.cross(rs_Raro, P_i)

L_tots_vec = np.sum(L_i, axis=0)

# CORRECCIÓN: L_tots_vec ya contiene el término del CM porque rs_Raro son posiciones ABSOLUTAS
# La fórmula es: L_tot = sum_i m_i(r_i' x v_i') + M(R_cm x V_cm)
# Y L_tots_vec ya incluye ambos términos

print('El momento angular total del sistema (plano invariante de Laplace) es:')
print('Vector:', L_tots_vec[0]* Msun2kg*UA2km/yr2s, 'kg*km^2/s')
print('Magnitud:', np.linalg.norm(L_tots_vec[0])* Msun2kg*UA2km/yr2s, 'kg*km^2/s')

In [ ]:
plt.plot(ts[:10], np.linalg.norm(np.cross(R_cm_vec, P_tots_vec)[:10], axis=1)* Msun2kg*UA2km/yr2s)

In [ ]:
# CORRECCIÓN: El plano invariante de Laplace ES L_tots_vec
# No debe sumarse nuevamente el término del centro de masa porque rs_Raro ya son posiciones ABSOLUTAS
# y L_tots_vec ya contiene: sum_i m_i(r_i' x v_i') + M(R_cm x V_cm)

# Si deseas descomponer en componentes:
L_relativo_al_CM = L_tots_vec - np.cross(R_cm_vec, P_tots_vec)  # Momentum angular relativo al CM
L_del_CM = np.cross(R_cm_vec, P_tots_vec)  # Momentum angular del centro de masa

# El plano invariante de Laplace es el momentum angular total (L_tots_vec)
L_Laplace = L_tots_vec  # Este es el vector correcto del plano invariante

L_Laplace_magnitud = np.linalg.norm(L_Laplace, axis=1)

print('Descomposición del momentum angular:')
print('L (relativo al CM):', np.linalg.norm(L_relativo_al_CM[0])* Msun2kg*UA2km/yr2s, 'kg*km^2/s')
print('L (del CM):', np.linalg.norm(L_del_CM[0])* Msun2kg*UA2km/yr2s, 'kg*km^2/s')wwwwwwwwwwwwwwwww
print('L (total/Laplace):', np.linalg.norm(L_Laplace[0])* Msun2kg*UA2km/yr2s, 'kg*km^2/s')

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(ts, np.linalg.norm(L_Laplace, axis=1)* Msun2kg*UA2km/yr2s, label='L total (Laplace)', linewidth=2)
plt.plot(ts, np.linalg.norm(L_relativo_al_CM, axis=1)* Msun2kg*UA2km/yr2s, label='L relativo al CM', linewidth=2)
plt.plot(ts, np.linalg.norm(L_del_CM, axis=1)* Msun2kg*UA2km/yr2s, label='L del CM', linewidth=2)
plt.xlabel('Tiempo (años)')
plt.ylabel('Momentum Angular (kg*km²/s)')
plt.title('Conservación del Plano Invariante de Laplace')
plt.legend()
plt.grid()
plt.show()

Energia mecánica Total del sistema $E$:

$$
E = \frac{1}{2}\sum_i m_i \dot{\vec{r}}^2_i  - \sum_i \sum_{j > i} G \frac{m_i m_j}{r_{ij}}  = K + U
$$

Calculo de la energia cinética:

$
K = \frac{1}{2}\sum_i m_i \dot{\vec{r}}^2_i
$

In [ ]:
v_2 = v_squared = np.sum(vs_Raro**2, axis=2)

Ks = 0.5 * ms[:, np.newaxis] * v_2

K = np.sum(Ks, axis=0)    # energia cinetica total del sistema 

Calculo de la energia potencial del sistema:

$
U =  - \sum_i \sum_{j < i} G \frac{m_i m_j}{r_{ij}}
$

In [ ]:
G_SI = 6.67430e-11  # m^3 kg^-1 s^-2

G = G_SI * (Msun2kg) * (UA2km*1000)**(-3) * (yr2s)**(2)  # Convertir a unidades de Msun, AU, yr

U_i = []
for i in range(len(sim_solar.particles) - 1):
    u_i = 0
    for j in range(i + 1, len(sim_solar.particles)):
        r_ij = rs_Raro[i] - rs_Raro[j]
        r_ij_norm = np.linalg.norm(r_ij, axis=1)
        U_ij = - G * ms[i] * ms[j] / r_ij_norm
        u_i += U_ij
    U_i.append(u_i)



U = np.sum(U_i, axis=0)


ENcontremos la energia mecanica total:

$
E = K + U
$

In [ ]:
E = K + U
E[:10]

In [ ]:
plt.plot(ts, abs(E[0] - E), label='Energía total del sistema (referencia al tiempo inicial)')
plt.xlabel('Tiempo')
plt.ylabel('Energía')
plt.title('Conservación de la Energía en el Sistema Solar')
plt.legend()
plt.show()


### Analisis del Virial: 

$$
<K> + <E> = 0
$$

In [ ]:
K_prom = np.mean(K)
E_prom = np.mean(E)

K_prom + E_prom

## **Análisis usando la teoria de 2 cuerpos** 

En esta seccion se analizará la dinamica de aphopis mediante las posiciones encontradas en la integracion 

In [ ]:
rs_Raro[1, 0, :], vs_Raro[1, 0, :] 

In [ ]:
M = np.sum(ms)

In [ ]:
mu = G * M
r_rel_1 = rs_Raro[1,0] #- rs_Raro[2,0]  # vector posición Apophis - sol
v_rel_1 = vs_Raro[1,0] #- vs_Raro[2,0]  # vector velocidad Apophis - sol

r_norm = np.linalg.norm(r_rel_1, axis=0)
v_norm = np.linalg.norm(v_rel_1, axis=0)

h = np.cross(r_rel_1, v_rel_1)
ener_rel = 0.5 * v_norm**2 - mu / r_norm
e_vec = ((np.cross(v_rel_1, h))/mu - (r_rel_1 / r_norm))

e = np.linalg.norm(e_vec)

print(f"Excentricidad de la órbita de Apophis respecto al Sol: {e:.6f}")
print(f"El momento angular específico de Apophis respecto al Sol es: {np.linalg.norm(h)* UA2km**2/yr2s} km^2/s")
print(f"La energía específica de Apophis respecto al Sol es: {ener_rel * UA2km**2/yr2s} km^2/s^2")


In [ ]:
import spiceypy as spy  

In [ ]:
spy.oscelt(list(rs_Raro[1, 0, :]) + list(vs_Raro[1, 0, :]), 0, G*M)


In [ ]:
h = np.cross(r_rel_1, v_rel_1)
ener_rel = 0.5 * v_norm**2 - mu / r_norm
e_vec = ((np.cross(v_rel_1, h))/mu - (r_rel_1 / r_norm))

e = np.linalg.norm(e_vec)
n_vec = np.cross(np.array([0, 0, 1]), h)

I = np.arccos(h[2] / np.linalg.norm(h))

# Determinacion del nodo ascendente

if n_vec[1] >= 0:
    Omega = np.arccos(n_vec[0] / np.linalg.norm(n_vec))

else:
    Omega = 2 * np.pi - np.arccos(n_vec[0] / np.linalg.norm(n_vec))

# Determinacion del argumento del perihelio

if e_vec[2] >= 0:
    omega = np.arccos(np.dot(n_vec, e_vec) / (np.linalg.norm(n_vec) * np.linalg.norm(e_vec)))
else:
    omega = 2 * np.pi - np.arccos(np.dot(n_vec, e_vec) / (np.linalg.norm(n_vec) * np.linalg.norm(e_vec)))

# Anomalia verdadera

if np.dot(r_rel_1, v_rel_1) >= 0:
    nu = np.arccos(np.dot(e_vec, r_rel_1) / (e * r_norm))

else:
    nu = 2 * np.pi - np.arccos(np.dot(e_vec, r_rel_1) / (e * r_norm))


print(f"Elementos orbitales de Apophis respecto al barycenter:")
print(f"Semieje mayor: {mu / (2 * abs(ener_rel)):.4f} AU")
print(f"Semilatus rectum: {np.linalg.norm(h)**2 / mu:.4f} AU")
print(f"Excentricidad: {e:.4f}")
print(f"Inclinación: {np.degrees(i):.4f} grados")
print(f"Nodo ascendente: {np.degrees(Omega):.4f} grados")
print(f"Argumento del perihelio: {np.degrees(omega):.4f} grados")
print(f"Anomalía verdadera: {np.degrees(nu):.4f} grados")

### **Dibujando la orbita sin rotacion, ni dependencia temporal**

In [ ]:
# grafica de la conica correspondiente a la orbita en la fecha inicial de apophis

fs = np.linspace(0, 2*np.pi, 1000)
r_conica = (np.linalg.norm(h)**2 / mu) / (1 + e * np.cos(fs))

plt.figure(figsize=(8, 8))
plt.plot(r_conica * np.cos(fs), r_conica * np.sin(fs), label='Órbita de Apophis (conica)', color='red')
plt.plot(rs_Raro[1, :, 0], rs_Raro[1, :, 1], color='blue', label='Tierra')
plt.scatter(0, 0, color='yellow', s=100, label='Sol')
plt.xlabel('X (AU)')
plt.ylabel('Y (AU)')
plt.title('Órbita de Apophis respecto al Sol (conica)')
plt.legend('lower left')
plt.axis('equal')
plt.grid()
plt.show()

### **Dibujando la orbita teniendo en cuenta las rotaciones y no la parte temporal**

In [ ]:
from funciones_utiles import rotacion_peri_astronomico

x_conica = r_conica * np.cos(fs)
y_conica = r_conica * np.sin(fs)
z_conica = np.zeros_like(x_conica)

vx_conica = - mu/np.linalg.norm(h) * np.sin(fs)
vy_conica = mu/np.linalg.norm(h) * (e + np.cos(fs))
vz_conica = np.zeros_like(vx_conica)

rs_conica = np.vstack((x_conica, y_conica, z_conica)).T
vs_conica = np.vstack((vx_conica, vy_conica, vz_conica)).T

rs_conica.shape

In [ ]:
# rotacion_peri_astronomico suele trabajar con arrays de forma (3, N)
rs_rel_real, vs_rel_real = rotacion_peri_astronomico(
    rs_conica.T,
    vs_conica.T,
    Omega,
    I,
    omega
)


# Si después necesitas volver a (N, 3):

rs_rel_real = rs_rel_real.T
vs_rel_real = vs_rel_real.T

In [ ]:
plt.figure(figsize=(8, 8))

plt.plot(rs_rel_real[:100, 0], rs_rel_real[:100, 1], label='Órbita de Apophis (conica)', color='red')
plt.plot(rs_Raro[1, :, 0], rs_Raro[1, :, 1], color='blue', label='Tierra')
plt.scatter(0, 0, color='yellow', s=100, label='Sol')
plt.xlabel('X (AU)')
plt.ylabel('Y (AU)')
plt.title('Órbita de Apophis respecto al Sol (conica)')
plt.legend('lower left')
plt.axis('equal')
plt.grid()
plt.show()

### **Ahora, dibujemos la orbita teniendo en cuenta la dependencia temporal de $f$**

$M = M_{ini} + n (t-t_{ini})$

donde:

$
M_{ini} = E_{ini} - e \sin{E_{ini}}
$

y 

$
\tan{\frac{f_{ini}}{2}} = \sqrt{\frac{1 + e}{1 - e}} \tan{\frac{E_{ini}}{2}}
$

In [ ]:
from scipy.optimize import newton

E_ini = 2 *np.arctan2(np.sqrt(1 - e) * np.sin(nu/2),np.sqrt(1 + e) * np.cos(nu/2))  # Anomalía excéntrica inicial

M_ini = E_ini - e * np.sin(E_ini)  # Anomalía media inicial 


M = lambda n, t, M_ini, tp: n * (t - tp)  +  M_ini                     # Anomalía media
n = np.sqrt(mu / (mu / (2 * abs(ener_rel)))**3)         # n = sqrt(mu/a^3)



M_conica = M(n, ts, M_ini, 0)  # Anomalía media a lo largo del tiempo, con M_ini = 0 para simplificar

# definamos la funcion para resolver la ecuacion de Kepler 

def kepler_equation(E, M, e):
    return E - e * np.sin(E) - M

E_conica = np.zeros_like(M_conica)

for i in range(len(M_conica)):
    E_conica[i] = newton(kepler_equation, M_conica[i], args=(M_conica[i], e))

# Ahora podemos calcular la anomalia verdadera a partir de la anomalia excéntrica:

fs_ver = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E_conica / 2), np.sqrt(1 - e) * np.cos(E_conica / 2))


In [ ]:
r_conica_ver = (np.linalg.norm(h)**2 / mu) / (1 + e * np.cos(fs_ver))

x_conica_ver = r_conica_ver * np.cos(fs_ver)
y_conica_ver = r_conica_ver * np.sin(fs_ver)
z_conica_ver = np.zeros_like(x_conica_ver)

vx_conica_ver = - mu/np.linalg.norm(h) * np.sin(fs_ver)
vy_conica_ver = mu/np.linalg.norm(h) * (e + np.cos(fs_ver))
vz_conica_ver = np.zeros_like(vx_conica_ver)

rs_conica_ver = np.vstack((x_conica_ver, y_conica_ver, z_conica_ver)).T
vs_conica_ver = np.vstack((vx_conica_ver, vy_conica_ver, vz_conica_ver)).T

rs_conica_ver.shape




In [ ]:
plt.figure(figsize=(8, 8))
plt.plot(x_conica_ver, y_conica_ver, label='Órbita de Apophis teoria de los dos cuerpos (verificada)', color='green')

In [ ]:
rs_rel_real_real, vs_rel_real_real = rotacion_peri_astronomico(
    rs_conica_ver.T,
    vs_conica_ver.T,
    Omega,
    I,
    omega
)

rs_rel_real_real = rs_rel_real_real.T
vs_rel_real_real = vs_rel_real_real.T


In [ ]:
plt.figure(figsize=(8, 8))
plt.plot(rs_rel_real_real[:, 0], rs_rel_real_real[:, 1], label='Órbita de Apophis teoria de los dos cuerpos', color='red')
plt.plot(rs_Raro[1, :, 0], rs_Raro[1, :, 1], color='blue', label='APophis (simulación numérica N cuerpos)')
plt.scatter(0, 0, color='yellow', s=100, label='Sol')
plt.xlabel('X (AU)')
plt.ylabel('Y (AU)')
plt.title('Órbita de Apophis respecto al Sol (conica)')
plt.legend()
plt.axis('equal')
plt.grid()
plt.show()

Diferencia de posiciones en el tiempo (Comparacion de N cuerpos con la teoria de 2 cuerpos)

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(ts, np.linalg.norm(rs_rel_real_real - rs_Raro[1, :, :], axis=1), label='Error de la órbita con la trayectoria real')
plt.xlabel('Tiempo (años)')
plt.ylabel('Error de la orbita con la trayectoria real (AU)')
plt.title('Distancia entre Tierra y Apophis a lo largo del tiempo')
plt.legend()
plt.grid()
plt.show()


In [ ]:
plt.figure(figsize=(8, 8))
plt.plot(ts, np.linalg.norm(rs_rel_real_real, axis=1), label='órbita')
plt.plot(ts, np.linalg.norm(rs_Raro[1], axis=1), label='trayectoria simulada con integrador', linestyle='--')
plt.plot(ts, np.linalg.norm(rs_JPL_Apo, axis=1), label='trayectoria JPL', linestyle='-.')
plt.xlabel('Tiempo (años)')
plt.ylabel('Distancia al Baricentro (AU)')
plt.title('Comparación entre la órbita teórica de Apophis y la trayectoria simulada')
plt.legend()
plt.grid()
plt.show()


## Análisis de la evolucion temporal de los elementos orbitales de Apophis

Para esto, vamos a usar la funcion `estado_a_elementos` del paquete **pymcel**

In [ ]:
# Hagamos el vector de estado para Apophis en el tiempo inicial, y luego lo convertimos a elementos orbitales usando la función de PyMcel

X_apos = np.hstack((rs_Raro[1], vs_Raro[1]))

X_apos.shape

In [ ]:
elementos = np.array([pym.estado_a_elementos(mu, X_apos[j]) for j in range(len(ts))])

elementos.shape

Guardemos los elementos orbitales de apophis 

In [ ]:
ps, es, Is, Ws, ws, fs = elementos.T

In [ ]:
# Graficar la evolución temporal de los 6 elementos orbitales de Apophis
fig, axes = plt.subplots(2, 3, figsize=(16, 14))
fig.suptitle('Evolución Temporal de los Elementos Orbitales de Apophis', fontsize=16, fontweight='bold')

# Subplot 1: Semieje mayor (p)
axes[0, 0].plot(ts, ps, color='blue', linewidth=2)
axes[0, 0].set_xlabel('Tiempo (años)')
axes[0, 0].set_ylabel('Semieje Mayor (AU)')
axes[0, 0].set_title('Semieje Mayor (a)')
axes[0, 0].grid(True, alpha=0.3)

# Subplot 2: Excentricidad (e)
axes[0, 1].plot(ts, es, color='red', linewidth=2)
axes[0, 1].set_xlabel('Tiempo (años)')
axes[0, 1].set_ylabel('Excentricidad')
axes[0, 1].set_title('Excentricidad (e)')
axes[0, 1].grid(True, alpha=0.3)

# Subplot 3: Inclinación (I) - convertir a grados
axes[0, 2].plot(ts, np.degrees(Is), color='green', linewidth=2)
axes[0, 2].set_xlabel('Tiempo (años)')
axes[0, 2].set_ylabel('Inclinación (grados)')
axes[0, 2].set_title('Inclinación (i)')
axes[0, 2].grid(True, alpha=0.3)

# Subplot 4: Longitud del nodo ascendente (W) - convertir a grados
axes[1, 0].plot(ts, np.degrees(Ws), color='purple', linewidth=2)
axes[1, 0].set_xlabel('Tiempo (años)')
axes[1, 0].set_ylabel('Longitud del nodo ascendente (grados)')
axes[1, 0].set_title('Longitud del Nodo Ascendente (Ω)')
axes[1, 0].grid(True, alpha=0.3)

# Subplot 5: Argumento del perihelio (w) - convertir a grados
axes[1, 1].plot(ts, np.degrees(ws), color='orange', linewidth=2)
axes[1, 1].set_xlabel('Tiempo (años)')
axes[1, 1].set_ylabel('Argumento del perihelio (grados)')
axes[1, 1].set_title('Argumento del Perihelio (ω)')
axes[1, 1].grid(True, alpha=0.3)

# Subplot 6: Anomalía verdadera (f) - convertir a grados
axes[1, 2].plot(ts, np.degrees(fs), color='brown', linewidth=2)
axes[1, 2].set_xlabel('Tiempo (años)')
axes[1, 2].set_ylabel('Anomalía verdadera (grados)')
axes[1, 2].set_title('Anomalía Verdadera (f)')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Veamos como varian los elementos orbitales de apophis del JPL

In [ ]:
# estados JPL (pos en AU, vel en AU/yr)
X_apo_jpl = np.column_stack((
    X_apo['x']/(UA2km*1000),
    X_apo['y']/(UA2km*1000),
    X_apo['z']/(UA2km*1000),
    X_apo['vx']*yr2s/(UA2km*1000),
    X_apo['vy']*yr2s/(UA2km*1000),
    X_apo['vz']*yr2s/(UA2km*1000),
))



X_apo_jpl.shape

In [ ]:
elementos_JPL = np.array([pym.estado_a_elementos(mu, X_apo_jpl[j]) for j in range(len(ts))])

elementos_JPL.shape

In [ ]:
ps_JPL, es_JPL, Is_JPL, Ws_JPL, ws_JPL, fs_JPL = elementos_JPL.T

In [ ]:
# Graficar la evolución temporal de los 6 elementos orbitales de Apophis
fig, axes = plt.subplots(2, 3, figsize=(16, 14))
fig.suptitle('Evolución Temporal de los Elementos Orbitales de Apophis', fontsize=16, fontweight='bold')

# Subplot 1: Semieje mayor (p)
axes[0, 0].plot(ts, ps_JPL, color='blue', linewidth=2)
axes[0, 0].set_xlabel('Tiempo (años)')
axes[0, 0].set_ylabel('Semilatus Rectum (AU)')
axes[0, 0].set_title('Semilatus Rectum (p)')
axes[0, 0].grid(True, alpha=0.3)

# Subplot 2: Excentricidad (e)
axes[0, 1].plot(ts, es_JPL, color='red', linewidth=2)
axes[0, 1].set_xlabel('Tiempo (años)')
axes[0, 1].set_ylabel('Excentricidad')
axes[0, 1].set_title('Excentricidad (e)')
axes[0, 1].grid(True, alpha=0.3)

# Subplot 3: Inclinación (I) - convertir a grados
axes[0, 2].plot(ts, np.degrees(Is_JPL), color='green', linewidth=2)
axes[0, 2].set_xlabel('Tiempo (años)')
axes[0, 2].set_ylabel('Inclinación (grados)')
axes[0, 2].set_title('Inclinación (i)')
axes[0, 2].grid(True, alpha=0.3)

# Subplot 4: Longitud del nodo ascendente (W) - convertir a grados
axes[1, 0].plot(ts, np.degrees(Ws_JPL), color='purple', linewidth=2)
axes[1, 0].set_xlabel('Tiempo (años)')
axes[1, 0].set_ylabel('Longitud del nodo ascendente (grados)')
axes[1, 0].set_title('Longitud del Nodo Ascendente (Ω)')
axes[1, 0].grid(True, alpha=0.3)

# Subplot 5: Argumento del perihelio (w) - convertir a grados
axes[1, 1].plot(ts, np.degrees(ws_JPL), color='orange', linewidth=2)
axes[1, 1].set_xlabel('Tiempo (años)')
axes[1, 1].set_ylabel('Argumento del perihelio (grados)')
axes[1, 1].set_title('Argumento del Perihelio (ω)')
axes[1, 1].grid(True, alpha=0.3)

# Subplot 6: Anomalía verdadera (f) - convertir a grados
axes[1, 2].plot(ts, np.degrees(fs_JPL), color='brown', linewidth=2)
axes[1, 2].set_xlabel('Tiempo (años)')
axes[1, 2].set_ylabel('Anomalía verdadera (grados)')
axes[1, 2].set_title('Anomalía Verdadera (f)')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Graficar la evolución temporal de los 6 elementos orbitales de Apophis
fig, axes = plt.subplots(2, 3, figsize=(16, 14))
fig.suptitle('Diferencia en la Evolución Temporal de los Elementos Orbitales de Apophis', fontsize=16, fontweight='bold')

# Subplot 1: Semilatus Rectum (p)
axes[0, 0].plot(ts, abs(ps - ps_JPL), color='blue', linewidth=2)
axes[0, 0].set_xlabel('Tiempo (años)')
axes[0, 0].set_ylabel('Semilatus Rectum (AU)')
axes[0, 0].set_title('Semilatus Rectum (p)')
axes[0, 0].grid(True, alpha=0.3)

# Subplot 2: Excentricidad (e)
axes[0, 1].plot(ts, abs(es - es_JPL), color='red', linewidth=2)
axes[0, 1].set_xlabel('Tiempo (años)')
axes[0, 1].set_ylabel('Excentricidad')
axes[0, 1].set_title('Excentricidad (e)')
axes[0, 1].grid(True, alpha=0.3)

# Subplot 3: Inclinación (I) - convertir a grados
axes[0, 2].plot(ts, abs(np.degrees(Is_JPL) - np.degrees(Is)), color='green', linewidth=2)
axes[0, 2].set_xlabel('Tiempo (años)')
axes[0, 2].set_ylabel('Inclinación (grados)')
axes[0, 2].set_title('Inclinación (i)')
axes[0, 2].grid(True, alpha=0.3)

# Subplot 4: Longitud del nodo ascendente (W) - convertir a grados
axes[1, 0].plot(ts, abs(np.degrees(Ws_JPL) - np.degrees(Ws)), color='purple', linewidth=2)
axes[1, 0].set_xlabel('Tiempo (años)')
axes[1, 0].set_ylabel('Longitud del nodo ascendente (grados)')
axes[1, 0].set_title('Longitud del Nodo Ascendente (Ω)')
axes[1, 0].grid(True, alpha=0.3)

# Subplot 5: Argumento del perihelio (w) - convertir a grados
axes[1, 1].plot(ts, abs(np.degrees(ws_JPL) - np.degrees(ws)), color='orange', linewidth=2)
axes[1, 1].set_xlabel('Tiempo (años)')
axes[1, 1].set_ylabel('Argumento del perihelio (grados)')
axes[1, 1].set_title('Argumento del Perihelio (ω)')
axes[1, 1].grid(True, alpha=0.3)

# Subplot 6: Anomalía verdadera (f) - convertir a grados
axes[1, 2].plot(ts, abs(np.degrees(fs_JPL) - np.degrees(fs)), color='brown', linewidth=2)
axes[1, 2].set_xlabel('Tiempo (años)')
axes[1, 2].set_ylabel('Anomalía verdadera (grados)')
axes[1, 2].set_title('Anomalía Verdadera (f)')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## problema dinamico de los dos cuerpos para Apophis

In [ ]:
mus = G*ms[2] + G*ms[0] + G*ms[1]

X_apo = np.concatenate((rs_Raro[1, 0], vs_Raro[1, 0]))

In [ ]:
p_apo, e_apo, i_apo, W_apo, w_apo, f_apo = pym.estado_a_elementos(mus, X_apo)

Calculo de parametros de interes a partir de los elementos orbitales

In [ ]:
h_apo = np.sqrt(mus*p_apo)
a_apo = 0.5*(p_apo/ (1-e_apo) + p_apo/ (1+e_apo) )
b_apo = a_apo * np.sqrt(1-e_apo**2)

from astropy.time import Time

t = Time("2029-04-13 18:52:00", scale='tdb').jd / 365.25  # Convertir a años julianos
tp = Time("2026-04-25 00:00:00", scale='tdb').jd / 365.25  # Convertir a años julianos

Definimos la ecuacion de kepler:

$
E - e \sin{E}
$

In [ ]:
def ecuacion_kepler(E, h, a, b, t, tp):
    return E - e_apo * np.sin(E) - h/(a*b)*(t - tp)

ENcontramos la anomalia excentrica $E$ para la fecha del acercamiento minimo:

In [ ]:
from scipy.optimize import newton

E = newton(ecuacion_kepler, x0=np.pi, args=(h_apo, a_apo, b_apo, t, tp))



In [ ]:
# vamos a usar aproximaciones analiticas para calcular la anomalia excentrica 

n = h_apo / (a_apo * b_apo)  # velocidad angular media
M = n * (t - tp)  # Anomalía media



### Formulas aproximadas 

- $
 E = M + e \sin{M}
$ 
- $ E = M + e\sin{M} + \frac{e^2}{2} \sin{2M}$



In [ ]:
E3 = M + e_apo * np.sin(M) + (e_apo**2 / 2) * np.sin(2*M) + (e_apo**3 / 8) * (3*np.sin(3*M) - np.sin(M))
E2 = M + e_apo * np.sin(M) + (e_apo**2/ 2) * np.sin(2*M)
E1 = M + e_apo * np.sin(M)

print(f"Anomalía excéntrica exacta (Newton): {E:.6f} rad")
print(f"Anomalía excéntrica aproximada (3er orden): {E3:.6f} rad")
print(f"Anomalía excéntrica aproximada (2do orden): {E2:.6f} rad")
print(f"Anomalía excéntrica aproximada (1er orden): {E1:.6f} rad")

usando la relacion:

$$
\tan{\frac{f}{2}} = \sqrt{\frac{1 + e}{1 - e}} \tan{\frac{E}{2}}
$$

In [ ]:
f = 2 * np.arctan(np.sqrt((1+e_apo)/(1-e_apo)) * np.tan(E/2))
f